In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset,DataLoader

from tqdm import tqdm

SR = 22050
DURATION = 30
model_name = "cnn"
try:
    dir_path = f"/kaggle/working/{model_name}"
    os.makedirs(dir_path, exist_ok=True)
    print(f"Directory created at: {dir_path}")
except Exception as e:
    print(f"Error creating directory: {e}")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kgg_key")
secret_value_1 = user_secrets.get_secret("kgg_user")
secret_value_2 = user_secrets.get_secret("WANDB_API_KEY")


#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")

Directory created at: /kaggle/working/cnn
🚀 Using device: cuda
GPU is available.Setting RANDOM_SEED .... 
   GPU: Tesla T4
   Memory: 14.6 GB

✅ Environment setup complete!


First Neural Network & CNNs!

* Learn PyTorch basics: Tensors, Dataset (custom loader for training), DataLoader.
* Convert audio to 2D/1D Mel-Spectrograms.
* Build a simple CNN (Convolutional Neural Network)/NN (Neural Network) to process the spectrograms.
* Implement training loop, loss, optimizer, and wandb logging.
* Train and evaluate your CNN/NN (Neural Network)model.

# Definition

## - Utility

In [3]:
def extract_paths(pc=15000,test_size=0.2):
    GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
    train_paths = []
    val_paths = []
    count = 0
    tr_count = 0
    val_count = 0
    
    paths_count = pc
    test_size = paths_count*test_size
    train_size = paths_count - test_size
    
    for g in GENRES:
        root_dir_path = f"/kaggle/input/datasets/akashkumbhakar/{g}-mel-15000"
        for i in range(0,paths_count):
            file_name = f"mashup_{i}.wav.npy"
            path = os.path.join(root_dir_path,file_name)
            if i >= train_size:
                val_paths.append((path,g))
                val_count += 1
            else : 
                train_paths.append((path,g))
                tr_count += 1
            count += 1
    print("Total paths (music files) : ", count)
    print("Total training files : ", tr_count)
    print("Total validation files : ", val_count)
    return train_paths,val_paths

def check_split():
    GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
    g_c = {}
    for i in tr:
        if i[1] in g_c.keys():
            g_c[i[1]] += 1
        else:
            g_c[i[1]] = 1
    return g_c

def genre_to_idx(str_label):
    genre_to_id = {'blues':0, 'classical':1, 'country':2, 'disco':3, 'hiphop':4,'jazz':5, 'metal':6, 'pop':7, 'reggae':8, 'rock':9}
    y = genre_to_id[str_label]

    return y

def idx_to_genre(targets):
    id_to_genre = {0:'blues', 1:'classical', 2:'country', 3:'disco', 4:'hiphop',5:'jazz', 6:'metal', 7:'pop', 8:'reggae', 9:'rock'}
    y = [id_to_genre[id] for id in targets]

    return y    

## - Dataset and DataLoader

In [5]:
class MelDataset(Dataset):
    def __init__(self,paths,transform):  # paths : List[(path,label)]
        self.paths = paths
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self,idx):
        mel = np.load(self.paths[idx][0])
        label = genre_to_idx(self.paths[idx][1])
        #mel = torch.from_numpy(mel).float()
        mel = self.transform(mel)
        label = torch.tensor(label, dtype=torch.long)
        return mel,label

data_transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[-45.60], std=[16.60])
])

## - Coonfig

In [6]:
config = {
    "batch_size" : 64,
    "lr" : 0.0001,
    "epochs" : 20
}

In [8]:
train_paths,val_paths = extract_paths(pc=15000)

train_dataset = MelDataset(train_paths,data_transformer)
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)
val_dataset = MelDataset(val_paths,data_transformer)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

print(f"✅ BATCH SIZE : {config['batch_size']} | Size of Train Dataloader : {len(train_loader)}  |  Size of Val Dataloader : {len(val_loader)}")

Total paths (music files) :  150000
Total training files :  120000
Total validation files :  30000
✅ BATCH SIZE : 64 | Size of Train Dataloader : 1875  |  Size of Val Dataloader : 469


## - Model

In [9]:
class MelCNN(nn.Module):
    def __init__(self, num_classes):
        super(MelCNN, self).__init__()
        # First convolutional block
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        # Fully connected head
        self.fc1 = nn.Linear(64, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):

       # INPUT :  (B,1,64,647)

        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

## - Training

In [10]:
def train(model,train_loader,val_loader,loss_fn, optimizer, num_epochs=5):
    train_loss = []
    train_f1_score = []
    max_f1 = 0.0
    best_model_state = None
    model.train()

    # initialization of wandb
    run = wandb.init(
        entity="23f1001065-indian-institute-of-technology-madras",
        project="23f1001065-t12026",
        name="cnn3-10000-adam-lr-0.0001",
        config = config
    )
    for epoch in range(num_epochs):
        running_loss = 0.0
        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader),desc=f"Epoch {epoch+1}/{num_epochs}")
        for i,(mels,labels) in progress_bar:
            mels , labels = mels.to(device,non_blocking=True), labels.to(device,non_blocking=True)
            optimizer.zero_grad()
            
            outputs = model(mels)
            loss = loss_fn(outputs, labels)
        
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            avg_loss = running_loss / (i + 1)

            progress_bar.set_postfix({
                'loss': f"{avg_loss:.4f}"
            })
        loss = running_loss / len(train_loader)
        t,p,f1score,val_loss = validation(model,val_loader,criterion)
        print(f"Train Loss: {loss:.4f}")
        # Upload metrics
        run.log({
            "train_loss" : float(loss),
            "val_loss" : float(val_loss),
            "f1_score" : float(f1score)
        })
        if f1score > max_f1 + 0.001 :
            max_f1 = f1score
            best_model_state = model.state_dict()
            
    # Close Wandb
    run.finish()
    return best_model_state


## - Evaluation

In [11]:
def validation(model,val_loader,loss_fn):
    model.eval()
    val_loss = 0.0
    all_true = []
    all_pred = []
    with torch.no_grad():
        progress_bar = tqdm(enumerate(val_loader), total=len(val_loader),desc=f"Validation")
        for i,(mels,labels) in progress_bar:
            mels,labels = mels.to(device,non_blocking=True),labels.to(device,non_blocking=True)
            output = model(mels)
            loss = loss_fn(output,labels)

            probs = torch.softmax(output,dim=1) 
            predicted_y = torch.argmax(probs,dim=1)
            
            val_loss += loss.item()
            avg_loss = val_loss / (i + 1)

            progress_bar.set_postfix({
                'loss': f"{avg_loss:.4f}"
            })

            all_true.append(labels)
            all_pred.append(predicted_y)
        all_true = torch.cat(all_true,dim=0).cpu().numpy()
        all_pred = torch.cat(all_pred,dim=0).cpu().numpy()
        print(all_true.shape,all_pred.shape)
    
        loss = val_loss/len(val_loader)
        f1score = f1_score(all_true,all_pred,average='macro')

        print(f"Validation Loss : {loss},  F1_score : {f1score}")
        return all_true,all_pred,f1score,loss
            

# Wandb login

In [12]:
import wandb
print(f"Wandb version: {wandb.__version__}")
wandb.login(key=secret_value_2)

Wandb version: 0.24.0


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1001065 (23f1001065-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Training

In [13]:
model = MelCNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])

best_model = train(model,train_loader,val_loader,criterion, optimizer, num_epochs=config['epochs'])

Validation: 100%|██████████| 469/469 [01:17<00:00,  6.06it/s, loss=1.1098]


(30000,) (30000,)
Validation Loss : 1.1097616560931907,  F1_score : 0.6232624718663741
Train Loss: 1.5459


Validation: 100%|██████████| 469/469 [01:16<00:00,  6.16it/s, loss=0.8101]


(30000,) (30000,)
Validation Loss : 0.8100931338155701,  F1_score : 0.7169167273837987
Train Loss: 0.9287


Validation: 100%|██████████| 469/469 [01:17<00:00,  6.08it/s, loss=0.7590]


...

Validation: 100%|██████████| 469/469 [01:11<00:00,  6.56it/s, loss=0.3648]


(30000,) (30000,)
Validation Loss : 0.3648153011605684,  F1_score : 0.8675150337296286
Train Loss: 0.3450


Validation: 100%|██████████| 469/469 [01:07<00:00,  6.96it/s, loss=0.3145]

(30000,) (30000,)
Validation Loss : 0.31445484713260047,  F1_score : 0.8935911630966645
Train Loss: 0.3360


f1_score,▁▃▄▅▅▅▅▆▆▆▇▇▇▇█▇▇█▇█
train_loss,█▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▅▅▄▃▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁
f1_score,0.89359
train_loss,0.336
val_loss,0.31445


In [16]:
b_model = MelCNN(10).to(device)
b_model.load_state_dict(best_model)

<All keys matched successfully>

In [17]:
validation(b_model,val_loader,criterion)

Validation: 100%|██████████| 469/469 [00:27<00:00, 16.84it/s, loss=0.3144]

(30000,) (30000,)
Validation Loss : 0.3143508564561669,  F1_score : 0.8935911630966645


(array([4, 4, 1, ..., 1, 5, 2]),
 array([4, 4, 1, ..., 1, 5, 2]),
 0.8935911630966645,
 0.3143508564561669)

# Saving model

In [18]:
if os.path.exists(f"/kaggle/working/{model_name}"):
    torch.save(best_model, f"/kaggle/working/{model_name}/model.pth")
    print(f"✅ {model_name} saved.")
else:
    print(f"Path not exists.")

✅ cnn saved.


# Uploading to KaggleHub

In [19]:
import kagglehub

# Replace with path to directory containing model files.
LOCAL_MODEL_DIR = f'/kaggle/working/{model_name}'

MODEL_SLUG = model_name # Replace with model slug.

# Learn more about naming model variations at
# https://www.kaggle.com/docs/models#name-model.
VARIATION_SLUG = 'default' # Replace with variation slug.

kagglehub.model_upload(
  handle = f"akashkumbhakar/{MODEL_SLUG}/pyTorch/{VARIATION_SLUG}",
  local_model_dir = LOCAL_MODEL_DIR,
  version_notes = 'Update 2026-03-08')

Uploading Model https://api.kaggle.com/models/akashkumbhakar/cnn/pyTorch/default ...
Starting upload for file /kaggle/working/cnn/model.pth


Uploading: 100%|██████████| 140k/140k [00:00<00:00, 272kB/s]

Upload successful: /kaggle/working/cnn/model.pth (137KB)


Your model instance version has been created.
Files are being processed...
See at: https://api.kaggle.com/models/akashkumbhakar/cnn/pyTorch/default
